# Face Recognition: test notebook

A step-by-step playground for the `facerec` package. Use it to check that your photos enroll correctly,
tune the threshold, and try live recognition. For everyday use run the desktop app instead: `python app.py`.

**How it works**
1. **Detection**: *YuNet* finds faces in an image, plus 5 landmarks per face (eyes, nose, mouth corners).
2. **Alignment + embedding**: *SFace* uses the landmarks to align the face, then turns it into a **128-number vector** (an "embedding"). Photos of the same person give similar vectors.
3. **Recognition**: a new face's vector is compared to the stored vectors with **cosine similarity**. If the best match is above a threshold (default `0.363`), that person's name is shown. Otherwise the face is **Unknown**.

No training is needed. Adding a person just means storing their embeddings.

**Data layout**: one folder per person in `known_faces/`, with a few photos of them:
```
known_faces/
    Alice/
        1.jpg
        2.jpg
    Bob/
        photo_a.png
```
Tips: use **3 to 10 photos per person** with different angles and lighting, the face clearly visible and not too small. If a photo has several faces, the largest one is used.

> Privacy: `known_faces/` and `face_db.npz` are git-ignored. Clear this notebook's outputs before committing, because they show face crops.

## Step 1: setup
Loads the models. They are downloaded automatically into `models/` on the first run.

In [ ]:
import time
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

import facerec as fr

KNOWN_DIR = fr.DEFAULT_KNOWN_DIR
DB_PATH = fr.DEFAULT_DB_PATH
CAMERA_INDEX = 0          # change if you have several cameras

detector, recognizer = fr.load_models()
print("OpenCV", cv2.__version__, "- models loaded")

## Step 2: build the database of known faces
Reads every photo in `known_faces/`, computes an embedding and saves everything to `face_db.npz`. **Re-run this cell whenever you add or remove photos.** The thumbnails show the aligned crops, so you can check the right face was picked.

Photos where the face is hard to detect (turned head, odd angle) are retried with a lower detection threshold and kept only if the face matches that person's clear photos. Cut-off, covered or wrong faces are left out, and the reason is printed.

In [ ]:
names, embeddings, crops, results = fr.build_database(KNOWN_DIR, detector, recognizer)
fr.save_database(DB_PATH, names, embeddings, results)
print(f"Saved {len(names)} embeddings to {DB_PATH.name}:", dict(Counter(names)))
for r in results:
    if r.status != "ok":
        print(f"  {r.key}: {r.note}")

if crops:
    n = len(crops); cols = min(n, 8); rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.9), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, crop, name in zip(axes.flat, crops, names):
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax.set_title(name, fontsize=9)
    plt.tight_layout(); plt.show()

## Step 3 (optional): sanity check, how similar are the known faces?
The same person should score high (often > 0.5) and different people should score low (< 0.3). If two different people score high, add more varied photos or raise the threshold.

In [ ]:
if len(embeddings) > 1:
    E = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    S = E @ E.T
    labels = [f"{n}#{i}" for i, n in enumerate(names)]
    plt.figure(figsize=(0.45 * len(names) + 3, 0.45 * len(names) + 2))
    plt.imshow(S, vmin=-0.2, vmax=1, cmap="viridis"); plt.colorbar(label="cosine similarity")
    plt.xticks(range(len(labels)), labels, rotation=90, fontsize=7)
    plt.yticks(range(len(labels)), labels, fontsize=7)
    plt.title(f"Pairwise similarity (threshold = {fr.COSINE_THRESHOLD})"); plt.tight_layout(); plt.show()

## Step 4 (optional): test on a single photo
Point `TEST_IMAGE` at any photo (ideally one **not** in `known_faces/`) to see what the model predicts.

In [ ]:
TEST_IMAGE = None   # e.g. Path("some_group_photo.jpg")
THRESHOLD = fr.COSINE_THRESHOLD   # raise (e.g. 0.45) if strangers get matched

if TEST_IMAGE is not None:
    img = fr.load_image(TEST_IMAGE)
    for face in fr.detect_faces(detector, img):
        emb, _ = fr.embed_face(recognizer, img, face)
        label, score = fr.identify(emb, names, embeddings, THRESHOLD)
        fr.draw_label(img, face, label, score)
        print(f"{label:>12}  {score:.3f}")
    plt.figure(figsize=(8, 8)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

## Step 5: live recognition
A window opens with your webcam. Each face gets a box: **green + name** if known, **red + Unknown** if not. The number is the similarity score. Press **Q** in the window to stop.

In [ ]:
names, embeddings = fr.load_database(DB_PATH)
print(f"Loaded {len(names)} embeddings for {len(set(names))} people")

cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW) if hasattr(cv2, "CAP_DSHOW") else cv2.VideoCapture(CAMERA_INDEX)
prev = time.time()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("Could not read from camera"); break
        for face in fr.detect_faces(detector, frame):
            emb, _ = fr.embed_face(recognizer, frame, face)
            label, score = fr.identify(emb, names, embeddings, THRESHOLD)
            fr.draw_label(frame, face, label, score)
        now = time.time(); fps = 1 / max(now - prev, 1e-6); prev = now
        cv2.putText(frame, f"FPS {fps:.0f}  (Q to quit)", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        cv2.imshow("Face Recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()